In [7]:
# PARTE 1 de 2
part1 = r"""---
title: 'Pipeline multilenguaje (Python, R, Julia) para el monitoreo de manglar en la CGSM (2013--2025)'
author: 'Lina Maria Quintero Fonseca'
date: '2026-03-30'
format:
  html:
    toc: true
    toc-depth: 3
    number-sections: true
    theme: cosmo
    code-fold: true
    self-contained: true
  pdf:
    toc: true
    number-sections: true
    documentclass: article
    geometry: margin=2.5cm
    fontsize: 12pt
---

# Introduccion y Justificacion

La Cienaga Grande de Santa Marta ---CGSM--- constituye el sistema lagunar costero mas extenso de Colombia y uno de los humedales de mayor relevancia ecologica en America Latina, gracias a que sus aproximadamente 86.310 hectareas de manglar cumplen funciones de regulacion hidrica, proteccion costera, captura de carbono azul y sostenimiento de la pesca artesanal de mas de 30.000 habitantes riberenos (Instituto de Investigaciones Marinas y Costeras [INVEMAR], 2024). Reconocida como sitio Ramsar desde 1998 y Reserva de Biosfera UNESCO desde 2000, la CGSM ha sido objeto de multiples figuras de proteccion que reconocen su importancia ecologica; sin embargo, desde la decada de los noventa el sistema ha experimentado una degradacion severa y ciclica de su cobertura de manglar, impulsada por la interrupcion del flujo hidrico tras la construccion de la carretera Cienaga--Barranquilla en los anos cincuenta, la hipersalinizacion resultante, la deforestacion para actividades agropecuarias y la intensificacion de los eventos ENSO ---especialmente La Nina---, de modo que aunque la reapertura de cinco canales hidraulicos entre 1996 y 1998 promovio una recuperacion parcial, la dinamica del ecosistema continua siendo inestable y los ciclos de degradacion y recuperacion no estan completamente caracterizados a alta resolucion espaciotemporal (INVEMAR, 2024).

En este contexto, el monitoreo de manglares es de vital importancia, y la tecnologia para su realizacion mediante teledeteccion ha evolucionado en la ultima decada, pasando de clasificaciones multitemporales con imagenes Landsat a escala regional (Giri et al., 2011; Hamilton y Casey, 2016) al uso de la coleccion Sentinel-2, cuya resolucion espacial de 10 metros y temporal de 5 dias ha mejorado sustancialmente la capacidad de detectar cambios fenologicos y perturbaciones a escala local. En Colombia, el INVEMAR ha realizado monitoreos sistematicos de la CGSM documentando los ciclos de muerte y recuperacion del manglar a traves de seis estaciones permanentes de campo ---cinco en el Complejo de Pajarales y una en el sector de Sevillano---, cuyos datos de estructura forestal estan publicados en el repositorio del Global Biodiversity Information Facility ---GBIF--- (Beltran et al., 2022; DOI: 10.15472/0fqdp4). No obstante, estos estudios se basan principalmente en interpretacion visual o clasificaciones supervisadas clasicas, sin recurrir al analisis automatico basado en modelos de fundacion como el Segment Anything Model ---SAM--- de Meta AI, adaptado para datos geoespaciales a traves de SamGeo (Wu y Osco, 2023), que permite realizar segmentacion semantica de imagenes satelitales sin necesidad de grandes conjuntos de datos etiquetados.

Asi, la integracion de herramientas de Inteligencia Artificial Geoespacial ---GeoAI--- con plataformas de geocomputacion en la nube como Google Earth Engine ---GEE--- representa una oportunidad para el monitoreo costero dinamico y de alta precision. El presente proyecto propone desarrollar un pipeline multilenguaje ---Python, R y Julia--- que integre el analisis de series de tiempo de indices espectrales con la segmentacion automatica de cobertura de manglar, constituyendo ademas un prototipo de componente operativo para un Digital Twin costero de la CGSM. La caracterizacion espaciotemporal del manglar en la CGSM constituye un insumo para la gestion de riesgos de inundacion y la formulacion de politica publica ambiental; en febrero de 2026 se firmo el Plan de Manejo Ambiental del sitio Ramsar Sistema Delta Estuarino del Rio Magdalena--CGSM, con una vigencia de diez anos, el cual requiere insumos espaciales actualizados para orientar las medidas de manejo y seguimiento del humedal (Comision Conjunta CGSM, 2026), de modo que un pipeline automatizado como el que aqui se propone atiende directamente ese requerimiento.

# Estado del Arte

## Teledeteccion de manglares: de Landsat al monitoreo multitemporal con Sentinel-2

El monitoreo satelital de manglares ha avanzado en las ultimas tres decadas, pasando de las clasificaciones multitemporales basadas en imagenes Landsat que permitieron cuantificar cambios historicos de cobertura a escala regional (Giri et al., 2011) al desarrollo de bases de datos globales continuas como la CGMFC-21, que proporciono estimaciones anuales de cobertura de manglar a resolucion de 30 metros para el periodo 2000--2012 (Hamilton y Casey, 2016). Con el lanzamiento de la coleccion Sentinel-2, la resolucion espacial de 10 metros y la revisita de 5 dias mejoraron sustancialmente la capacidad de detectar cambios fenologicos a escala local, de modo que hoy es posible construir series temporales densas que capturen la variabilidad estacional e interanual de la cobertura vegetal costera. Raza et al. (2024) demostraron que el analisis de series de tiempo del NDVI derivadas de Sentinel-2 en GEE permite detectar tendencias de deterioro en la salud del manglar incluso cuando la extension del bosque se mantiene estable, razon por la cual las estimaciones de cobertura por si solas resultan insuficientes y deben acompanarse de indicadores de condicion de la vegetacion.

En cuanto a los indices espectrales, Gupta et al. (2018) propusieron el Combined Mangrove Recognition Index ---CMRI---, definido como la diferencia entre el NDVI y el NDWI, que ha demostrado ser efectivo para discriminar manglar de otras coberturas vegetales en ambientes estuarinos, alcanzando una precision del 73,43% frente a indices convencionales como el NDVI (56,29%) o el Simple Ratio (48,79%). A escala global, el Global Mangrove Watch ---GMW--- de JAXA provee la referencia mas completa de distribucion de manglares para el periodo 1996--2020 en su version 3.0 (Bunting et al., 2022), mientras que a escala nacional, el INVEMAR genero en 2020 la cartografia oficial de manglares de Colombia a escala 1:25.000 mediante clasificacion supervisada en GEE con imagenes opticas y de radar, con una unidad minima cartografiada de 1.600 m2 (INVEMAR, 2020).

## Google Earth Engine como plataforma de monitoreo

La adopcion de GEE para el estudio de manglares se ha acelerado en los ultimos anos, pues la plataforma permite procesar miles de imagenes satelitales sin requerir infraestructura computacional local. Selvaraj y Gallego-Perez (2023) aplicaron GEE al mapeo de manglares en el Pacifico colombiano combinando datos opticos de Landsat y datos de radar SAR de ALOS-2/PALSAR-2 con un clasificador Random Forest, obteniendo precisiones moderadas a altas en la deteccion de cambios de cobertura entre 2009 y 2019. Bunting et al. (2022) utilizaron GEE como plataforma de procesamiento para generar la version 3.0 del GMW, que proporciona mapas anuales de extension de manglar a escala global para el periodo 1996--2020 combinando datos opticos y de radar.

## Monitoreo de la Cienaga Grande de Santa Marta

En la CGSM, el INVEMAR realiza monitoreo continuo desde la reapertura de los canales hidraulicos entre 1996 y 1998, evaluando la calidad de aguas, la estructura del bosque de manglar y los recursos pesqueros en el marco del Convenio de Cooperacion No. 16/2006 con la Corporacion Autonoma Regional del Magdalena ---CORPAMAG--- (INVEMAR, 2024). El monitoreo de manglar opera en seis estaciones permanentes: cinco en el Complejo de Pajarales ---Luna, Aguas Negras, Cano Grande, Km22 y Rinconada--- y una en el sector de Sevillano, y los datos de estructura forestal correspondientes al periodo 2013--2019 estan publicados en GBIF bajo licencia CC-BY (Beltran et al., 2022). El informe tecnico mas reciente documenta que la mayoria de las estaciones mantienen una categoria de integridad biologica "regular", con excepcion de Rinconada que se mantiene en "buen estado", y que el sector de Aguas Negras perdio el 33% de su arbolado por inundacion permanente (INVEMAR, 2024).

## GeoAI y segmentacion geoespacial

La aparicion de modelos de fundacion como el Segment Anything Model de Meta AI, adaptado para datos geoespaciales a traves de SamGeo (Wu y Osco, 2023), permite realizar segmentacion semantica de imagenes satelitales sin grandes conjuntos de datos etiquetados, mediante el uso de text prompts o puntos de referencia. La plataforma OpenGeoAI, liderada por el profesor Qiusheng Wu de la Universidad de Tennessee, ha desarrollado un ecosistema de herramientas de codigo abierto ---geemap, leafmap y SamGeo--- integradas con GEE, de manera que es posible construir pipelines completos de procesamiento geoespacial en la nube. El concepto de Digital Twin aplicado a ecosistemas costeros, explorado por Deng et al. (2021), combina datos de observacion de la Tierra, modelos fisicos y aprendizaje automatico para la simulacion dinamica de fenomenos costeros; en este marco, el componente de monitoreo basado en teledeteccion que este proyecto desarrolla constituye el modulo de observacion que alimentaria al gemelo digital con el estado actualizado del ecosistema.

# Objetivos

## Objetivo General

Desarrollar un pipeline GeoAI multilenguaje para el monitoreo de la dinamica espaciotemporal de la cobertura de manglar en la Cienaga Grande de Santa Marta (2013--2025).

## Objetivos Especificos

1. Construir un datacube multitemporal de indices espectrales para la caracterizacion de la cobertura de manglar en la CGSM.
2. Identificar los periodos de degradacion y recuperacion del manglar mediante analisis de anomalias temporales.
3. Validar la segmentacion automatica de cobertura de manglar contra cartografia de referencia.

# Area de Estudio

El area de estudio comprende la CGSM y su zona de influencia costera, ubicada en el departamento del Magdalena, Colombia, entre aproximadamente 10 20 N--11 05 N y 74 10 W--74 55 W. El poligono de analisis fue delimitado con 34 vertices cubriendo un area total de 5.073 km2, abarcando la Via Parque Isla de Salamanca ---barra costera con manglares protegidos que constituye zona nucleo de la Reserva de Biosfera---, el Complejo de Pajarales ---sistema de cienagas interconectadas donde se concentro la mayor degradacion historica del manglar---, el Santuario de Flora y Fauna CGSM con sus 26.810 hectareas de area protegida, los canales hidraulicos rehabilitados ---Cano Clarin, Aguas Negras y Renegado--- con conexion al rio Magdalena, y las desembocaduras de los rios provenientes de la Sierra Nevada de Santa Marta ---Sevilla, Aracataca y Fundacion--- (INVEMAR, 2024).

Para el analisis de series de tiempo se definieron 8 estaciones de muestreo, de las cuales 5 corresponden a estaciones con coordenadas exactas obtenidas del dataset de monitoreo de estructura de manglares del INVEMAR publicado en GBIF (Beltran et al., 2022; DOI: 10.15472/0fqdp4) ---Isla Boqueron (10,962 N, 74,298 W), Punta Cerro (10,973 N, 74,283 W), Punta Chino (10,912 N, 74,305 W), Rio Sevilla (10,880 N, 74,325 W) y Cano Palos (10,758 N, 74,471 W)---, mientras que las 3 restantes son estaciones complementarias seleccionadas sobre cobertura de manglar verificada mediante NDVI > 0,4 en composites Sentinel-2 de 2024, con el proposito de representar el Complejo de Pajarales y la zona de rehabilitacion hidrologica del Cano Clarin.

# Fuentes de Datos

Los datos utilizados en este proyecto provienen tanto de plataformas de geocomputacion en la nube como de repositorios institucionales abiertos.

| Dataset / Fuente | Tipo | Uso en el proyecto | Acceso |
|------------------|------|-------------------|--------|
| Sentinel-2 MSI L2A (ESA) | 789 imagenes (2018--2025) | Indices espectrales y RGB para SamGeo | GEE: COPERNICUS/S2_SR_HARMONIZED |
| Landsat 8 OLI (USGS) | 345 registros (2013--2017) | Serie historica NDVI complementaria | GEE: LANDSAT/LC08/C02/T1_L2 |
| Sentinel-1 SAR (ESA) | 85 imagenes (2020) | Deteccion de inundacion bajo dosel | GEE: COPERNICUS/S1_GRD |
| Global Flood Database | 16 eventos (2001--2017) | Registro historico de inundaciones | GEE: GLOBAL_FLOOD_DB |
| JRC Global Surface Water | Raster global | Transiciones y estacionalidad hidrica | GEE: JRC/GSW1_4 |
| SRTM v3 (NASA) | DEM 30 m | Restriccion de elevacion (< 10 m) | GEE: USGS/SRTMGL1_003 |
| Monitoreo manglar INVEMAR | 376 registros (2013--2019) | Coordenadas estaciones de muestreo | GBIF DOI: 10.15472/0fqdp4 |
| Global Mangrove Watch v4.0 | Raster (2020) | Validacion de clasificacion | GEE: sat-io/GMW |

: Fuentes de datos utilizadas en el pipeline de monitoreo de manglar. {#tbl-fuentes}

# Metodologia y Codigo

El proyecto se desarrolla siguiendo un pipeline modular y reproducible, organizado en cuatro fases y un modulo de validacion que se ejecutan dentro de un contenedor Docker ---sig_unal v1.11, que integra Python 3.12, R 4.3.3, Julia 1.11.3 y Quarto 1.4---, de modo que se garantiza la replicabilidad completa del entorno de analisis. Cada fase se implementa aprovechando las fortalezas del lenguaje mas adecuado para la tarea, y la interoperabilidad entre fases se mantiene mediante formatos estandar ---GeoTIFF, GeoJSON y CSV---.

## Fase 1: Construccion del datacube multitemporal (Python + geemap)

La primera fase del pipeline consiste en la construccion de un datacube listo para analisis ---una estructura tridimensional (X, Y, tiempo) que organiza las observaciones satelitales de la CGSM a resoluciones de 10 a 30 metros segun el sensor, donde cada capa temporal contiene los valores de reflectancia superficial e indices espectrales derivados para un periodo de composicion determinado---. Se construye una coleccion de 789 imagenes Sentinel-2 SR Harmonized para el periodo 2018--2025, filtrada por cobertura de nubes inferior al 20% y recortada al poligono del area de estudio. Se aplica una mascara de nubes utilizando la banda QA60 y se calculan los indices NDVI = (B8-B4)/(B8+B4), NDWI = (B3-B8)/(B3+B8) y CMRI = NDVI - NDWI por imagen, cuya efectividad para discriminar manglar de otras coberturas en ambientes estuarinos ha sido documentada por Gupta et al. (2018).
```python
# Filtrado y calculo de indices
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2018-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
      .map(add_indices))  # NDVI, NDWI, CMRI

def add_indices(image):
    ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])
```

## Fase 2: Analisis de series de tiempo (Python + pandas / R + bfast)

Se extraen series temporales mensuales de NDVI para las 8 estaciones de muestreo definidas, utilizando un buffer de 500 metros alrededor de cada punto y la funcion reduceRegion de GEE para obtener el valor medio del indice por estacion y por mes. La serie se construye combinando dos sensores: Landsat 8 Collection 2 Level 2 para el periodo 2013--2017 (345 registros, con correccion del factor de escala x0,0000275 - 0,2 segun especificacion USGS) y Sentinel-2 SR Harmonized para el periodo 2018--2025 (584 registros), generando una serie combinada de 929 observaciones mensuales que cubre 12 anos. Posteriormente se calcula el z-score temporal por estacion ---z = (NDVI del mes - media historica) / desviacion estandar---, lo que permite identificar anomalias pronunciadas definidas como aquellas con z < -2. De manera complementaria, se aplica el algoritmo bfast (Breaks For Additive Season and Trend) en R sobre las series mensuales combinadas con parametros h = 0,15 y h = 0,10 para evaluar la presencia de quiebres estructurales en la tendencia de largo plazo.
```python
# Z-scores y deteccion de anomalias
df['z_score'] = df.groupby('estacion')['ndvi'].transform(
    lambda x: (x - x.mean()) / x.std())
anomalias = df[df['z_score'] < -2].sort_values('z_score')
```

## Fase 3: Segmentacion automatica con SamGeo (Python + samgeo)

Se aplica el modelo SamGeo con el backbone vit_b sobre los composites RGB de los tres periodos de referencia ---degradacion (julio--diciembre 2020), recuperacion (enero--junio 2022) y estado actual (julio 2024 a junio 2025)---, previamente remuestreados de 10 a 30 metros para ajustarse a las limitaciones de memoria del contenedor Docker. Las mascaras resultantes se vectorizan y se clasifican por estado ---manglar o no manglar--- segun el valor medio de CMRI dentro de cada parche, calculado mediante reduceRegions en GEE con procesamiento en lotes de 200 parches. Se filtran los parches por area minima de 1 hectarea y maxima de 5.000 hectareas para eliminar fragmentos de ruido y fondos de imagen.
```python
from samgeo import SamGeo
sam = SamGeo(model_type='vit_b', automatic=True)
for nombre in ['degradacion', 'recuperacion', 'actual']:
    sam.generate(f'{res_dir}/CGSM_RGB_{nombre}_30m.tif',
                 output=f'{out_dir}/mask_{nombre}.tif')
    sam.tiff_to_vector(f'{out_dir}/mask_{nombre}.tif',
                       f'{out_dir}/manglar_{nombre}.geojson')
```

## Fase 4: Metricas de fragmentacion del paisaje (Julia)

Se computan metricas de ecologia del paisaje sobre los parches de manglar vectorizados para cada uno de los tres periodos, aprovechando la velocidad de Julia en operaciones geometricas sobre grandes volumenes de datos vectoriales. Las metricas incluyen: numero de parches, densidad de parches por 1.000 hectareas, area media de parche con desviacion estandar, indice de forma medio ---MSI = perimetro / sqrt(pi x area)---, distribucion de parches por clases de tamano (1--10, 10--50, 50--100, 100--500, 500--1.000 y 1.000--5.000 ha), y distancia media al vecino mas cercano ---NND--- como indicador de conectividad.
```julia
using GeoJSON, DataFrames, Statistics
function compute_metrics(periodo, base_dir)
    geojson = GeoJSON.read(read(path, String))
    # Shoelace formula para area, perimetro en metros
    # MSI = perimetro / sqrt(pi * area_m2)
    # NND = distancia minima entre centroides
    return Dict("n_parches" => n, "area_total" => sum(areas),
               "msi_mean" => mean(msi), "nnd_mean" => mean(nnd))
end
```

## Fase 5: Validacion con datos NASA y dashboard

Como componente de validacion cruzada, se integran datos de la Global Flood Database ---que registra 16 eventos de inundacion historicos en la CGSM entre 2001 y 2017--- y del JRC Global Surface Water ---que identifica 977,4 km2 de agua superficial permanente (ocurrencia > 50%) en el area de estudio---. Para el evento de septiembre 2020, se realiza una deteccion de inundacion mediante Sentinel-1 SAR (banda VH, modo IW), comparando el backscatter medio del periodo seco de referencia (enero--marzo 2020, 49 imagenes) contra el periodo de inundacion (septiembre--octubre 2020, 36 imagenes). Se aplica un umbral de diferencia de 3 dB para inundacion en agua abierta y se identifica inundacion bajo dosel de manglar mediante valores negativos de diferencia SAR ---donde el aumento del backscatter refleja el scattering de doble rebote caracteristico de la interaccion agua-tronco---.
```python
s1_dry = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterDate('2020-01-01','2020-03-31').select('VH').median()
s1_flood = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterDate('2020-09-01','2020-10-31').select('VH').median()
sar_diff = s1_dry.subtract(s1_flood)
flood_open = sar_diff.gt(3).selfMask()    # agua abierta
flood_canopy = sar_diff.lt(-2).selfMask() # bajo dosel
```

## Flujo de datos

| Paso | Herramienta | Lenguaje | Entrada | Salida |
|------|-------------|----------|---------|--------|
| 1. Adquisicion | GEE + geemap | Python | Sentinel-2 / Landsat | Stacks de indices (.tif) |
| 2. Series de tiempo | pandas + bfast | Python / R | Indices + estaciones | Z-scores, periodos (.csv) |
| 3. Segmentacion | SamGeo + umbrales | Python | Composites RGB | Parches manglar (.geojson) |
| 4. Metricas | DataFrames.jl | Julia | Parches vectoriales | Tabla de metricas (.csv) |
| 5. Inundacion SAR | GEE Sentinel-1 | Python | SAR VH seco vs humedo | Mapa inundacion (.tif) |
| 6. Dashboard | geemap | Python | Todas las capas | Dashboard (.html) |

: Flujo de datos del pipeline multilenguaje. {#tbl-flujo}

## Entorno de ejecucion

| Componente | Especificacion |
|-----------|----------------|
| Contenedor | Docker sig_unal v1.11 (Ubuntu + RStudio Server) |
| Python | 3.12.3 + geemap, samgeo, leafmap, rasterio, geopandas |
| R | 4.3.3 + terra, sf, tidyverse, bfast, tmap |
| Julia | 1.11.3 + DataFrames, CSV, GeoJSON, Statistics |
| Quarto | 1.4.550 (informe reproducible) |
| Control de versiones | Git + GitHub |
| Geocomputacion en la nube | Google Earth Engine (API Python) |

: Entorno de ejecucion del proyecto. {#tbl-entorno}
"""

with open('/home/rstudio/work/proyecto-cgsm/docs/informe_final.qmd', 'w') as f:
    f.write(part1)
print(f"Parte 1 escrita: {len(part1)} caracteres")

Parte 1 escrita: 21056 caracteres


In [8]:
part2 = r"""
# Resultados y Discusion

## Series temporales de NDVI y deteccion de anomalias

El analisis de series temporales de NDVI para las 8 estaciones de monitoreo revelo 18 anomalias significativas (z < -2) durante el periodo 2013--2025. El evento de septiembre de 2020 se identifico como la perturbacion de mayor magnitud, con valores negativos de NDVI en Punta Cerro (-0,078; z = -2,46) e Isla Boqueron (-0,018; z = -3,36), coincidente con el inicio de La Nina 2020--2021 que provoco inundaciones prolongadas en el sistema lagunar. Un segundo cluster de anomalias se concentro en marzo de 2018, afectando las estaciones occidentales ---Cano Clarin (0,166; z = -3,15) y CP Aguas Negras (0,203; z = -2,50)---, probablemente asociado a condiciones de hipersalinizacion en epoca seca. La extension de la serie con Landsat 8 revelo 4 anomalias adicionales en VIPIS durante 2016 (z = -2,93 en abril), que no eran visibles con la serie Sentinel-2 sola, demostrando la importancia de contar con series temporales largas. Las estaciones con mayor variabilidad fueron Cano Palos (std = 0,143) y Cano Clarin (std = 0,147), mientras que Punta Chino mostro la menor (std = 0,078), lo cual es consistente con su ubicacion protegida en el borde suroriental del sistema. Con base en estos hallazgos se seleccionaron tres periodos de referencia para la segmentacion: degradacion (julio--diciembre 2020), recuperacion (enero--junio 2022) y estado actual (julio 2024 a junio 2025).

| Estacion | Fecha | NDVI | z-score | Sensor |
|----------|-------|------|---------|--------|
| Isla Boqueron | 2020-09 | -0,018 | -3,36 | Sentinel-2 |
| Cano Clarin | 2018-03 | 0,166 | -3,15 | Sentinel-2 |
| VIPIS | 2016-04 | -0,291 | -2,93 | Landsat 8 |
| Cano Clarin | 2025-03 | 0,223 | -2,77 | Sentinel-2 |
| Cano Palos | 2024-09 | 0,272 | -2,66 | Sentinel-2 |
| CP Pajarales | 2018-03 | 0,203 | -2,50 | Sentinel-2 |
| Punta Cerro | 2020-09 | -0,078 | -2,46 | Sentinel-2 |
| CP Pajarales | 2022-09 | 0,217 | -2,40 | Sentinel-2 |

: Anomalias significativas de NDVI (z < -2) en la serie combinada 2013--2025. {#tbl-anomalias}

## Analisis bfast (serie combinada 2013--2025)

La aplicacion de bfast sobre las series mensuales de NDVI combinadas Landsat 8 + Sentinel-2 (2013--2025, 12 anos) detecto quiebres estructurales en 7 de las 8 estaciones con h = 0,15 y en las 8 estaciones con h = 0,10. Este resultado contrasta con el analisis previo basado unicamente en Sentinel-2 (2018--2025, 7 anos), que no detecto quiebres en ninguna estacion, lo cual demuestra la importancia de contar con series temporales suficientemente largas para la deteccion de cambios estructurales con bfast. El patron dominante es un quiebre generalizado en 2016 ---detectado en 7 de 8 estaciones entre abril y diciembre de ese ano---, coincidente con el evento El Nino 2015--2016, uno de los mas intensos registrados, que provoco sequias prolongadas y estres hidrico en el sistema lagunar. La estacion Punta Cerro fue la mas inestable, con 3 quiebres detectados (h = 0,10): noviembre de 2016 (El Nino), abril de 2020 (inicio de La Nina) y agosto de 2021 (recuperacion post-La Nina), capturando los dos eventos climaticos mas importantes del periodo de estudio. VIPIS presento un segundo quiebre en 2022, posiblemente asociado a la recuperacion post-La Nina 2020--2021.

| Estacion | Quiebres (h=0,15) | Fecha principal | Quiebres (h=0,10) | Evento asociado |
|----------|-------------------|-----------------|-------------------|-----------------|
| CP Pajarales | 1 | 2016-10 | 1 | El Nino 2015--2016 |
| Cano Clarin | 1 | 2016-09 | 1 | El Nino 2015--2016 |
| Cano Palos | 1 | 2015-06 | error | El Nino (inicio) |
| Isla Boqueron | 2 | 2016-04, 2018-05 | 2 | El Nino + transicion |
| Punta Cerro | 1 | 2016-11 | 3 | El Nino + La Nina + recuperacion |
| Punta Chino | 1 | 2016-05 | 2 | El Nino 2015--2016 |
| Rio Sevilla | 1 | 2016-12 | 1 | El Nino 2015--2016 |
| VIPIS | 2 | 2016-04, 2022-03 | 2 | El Nino + recuperacion |

: Quiebres estructurales detectados por bfast en la serie NDVI combinada (2013--2025). {#tbl-bfast}

## Segmentacion y comparacion multitemporal de cobertura de manglar

La segmentacion con SamGeo (vit_b, 30 m) y la clasificacion por umbrales espectrales (CMRI > 0) sobre los tres periodos de referencia arrojaron los siguientes resultados. El periodo actual (2024--2025) presenta el mayor numero de parches (34), la mayor extension de manglar (8.444,5 ha) y los valores mas altos de NDVI (0,564) y CMRI (1,081), lo que indica una recuperacion tanto en extension como en vigor vegetativo respecto al periodo de degradacion. Entre degradacion y recuperacion (2020--2022), el area aumento ligeramente de 4.938,5 a 5.067,9 hectareas, pero el NDVI disminuyo de 0,503 a 0,444, lo que sugiere una expansion espacial del manglar acompanada de una reduccion en el vigor fotosintetico, posiblemente asociada a rebrotes jovenes con menor biomasa foliar.

| Periodo | Parches | Area manglar (ha) | NDVI medio | CMRI medio |
|---------|---------|-------------------|------------|------------|
| Degradacion (2020-S2) | 16 | 4.938,5 | 0,503 | 0,954 |
| Recuperacion (2022-S1) | 16 | 5.067,9 | 0,444 | 0,882 |
| Actual (2024--2025) | 34 | 8.444,5 | 0,564 | 1,081 |

: Comparacion multitemporal de cobertura de manglar. {#tbl-multitemporal}

## Metricas de fragmentacion del paisaje

El analisis de fragmentacion computado en Julia revelo una tendencia clara de consolidacion del paisaje de manglar entre los tres periodos. La densidad de parches disminuyo de 11,19 a 6,47 parches por 1.000 hectareas entre el periodo de degradacion y el actual, mientras que el area media de parche aumento de 89,4 a 154,5 hectareas, lo que indica que los parches se estan fusionando en unidades mas grandes. La distancia media al vecino mas cercano ---NND--- aumento de 0,46 a 1,24 km, lo que refleja una redistribucion espacial de los parches con menor densidad pero mayor tamano individual. El indice de forma medio ---MSI--- paso de 0,29 a 0,74, indicando que los parches del periodo actual presentan formas mas complejas e irregulares que los del periodo de degradacion, un patron consistente con procesos de regeneracion natural que avanzan de manera no uniforme a lo largo de los bordes del manglar.

| Metrica | Degradacion | Recuperacion | Actual |
|---------|-------------|--------------|--------|
| Parches (1--5.000 ha) | 596 | 456 | 183 |
| Area total (ha) | 53.283,4 | 47.010,5 | 28.265,3 |
| Area media +/- std (ha) | 89,4 +/- 81,6 | 103,1 +/- 145,8 | 154,5 +/- 313,2 |
| Densidad (parches/1.000 ha) | 11,19 | 9,70 | 6,47 |
| MSI medio | 0,29 | 0,31 | 0,74 |
| NND medio (km) | 0,46 | 0,52 | 1,24 |

: Metricas de fragmentacion del paisaje de manglar por periodo. {#tbl-fragmentacion}

## Validacion con datos NASA: deteccion de inundacion SAR

La consulta a la Global Flood Database identifico 16 eventos de inundacion historicos en la CGSM entre 2001 y 2017, de los cuales 14 registraron areas de inundacion superiores a cero. El evento de mayor magnitud ocurrio en febrero de 2005 con 1.149,9 km2 inundados, seguido por los eventos de septiembre--diciembre de 2005 (1.063,7 km2, 92 dias de duracion ---el mas prolongado del registro---) y noviembre de 2004 (1.027,2 km2). Estos eventos coinciden con episodios ENSO documentados por el IDEAM, confirmando la vulnerabilidad del sistema lagunar a la variabilidad climatica interanual. Las estaciones del borde oriental ---Isla Boqueron, Punta Cerro y Rio Sevilla--- registraron inundacion en 10 de los 16 eventos, mientras que Cano Clarin solo fue afectado en 1 evento, lo cual es consistente con su ubicacion mas alejada del cuerpo de agua principal.

| Evento | Inicio | Fin | Area (km2) | Dias |
|--------|--------|-----|-----------|------|
| DFO_1996 | 2002-07-20 | 2002-07-31 | 959,5 | 11 |
| DFO_2588 | 2004-11-20 | 2004-11-27 | 1.027,2 | 7 |
| DFO_2625 | 2005-02-11 | 2005-02-26 | 1.149,9 | 15 |
| DFO_2761 | 2005-09-15 | 2005-12-16 | 1.063,7 | 92 |
| DFO_3212 | 2007-10-01 | 2007-12-10 | 1.025,7 | 70 |
| DFO_3750 | 2010-11-15 | 2010-12-20 | 946,6 | 35 |
| DFO_4495 | 2017-08-04 | 2017-08-21 | 908,9 | 17 |

: Eventos de inundacion con mayor area afectada en la CGSM (Global Flood Database, 2001--2017). {#tbl-gfd}

Para el evento de septiembre--octubre de 2020 ---no registrado en la Global Flood Database cuya cobertura termina en 2017---, la deteccion con Sentinel-1 SAR (banda VH, modo IW) identifico 56,6 km2 de inundacion en agua abierta (diferencia de backscatter > 3 dB) y 1.450,8 km2 de inundacion bajo dosel de manglar (diferencia negativa, indicativa de scattering de doble rebote agua-tronco), para un total de 1.507,4 km2 afectados ---el 29,7% del area de estudio---. El cruce con las anomalias NDVI revelo patrones diferenciados por estacion: Punta Cerro e Isla Boqueron presentaron degradacion severa (NDVI negativo) sin senal SAR de inundacion directa, lo que sugiere mortalidad de manglar por hipersalinizacion mas que por anegamiento; CP Pajarales mostro inundacion bajo dosel (SAR diff = -1,71 dB) con NDVI relativamente estable (0,300), indicando manglar inundado pero no muerto; y Rio Sevilla presento estres hidrico (NDVI = 0,050) sin inundacion SAR clara.

| Tipo de inundacion | Area (km2) | Porcentaje AOI |
|-------------------|-----------|----------------|
| Agua abierta (SAR diff > 3 dB) | 56,6 | 1,1% |
| Bajo dosel manglar (SAR diff < -2 dB) | 1.450,8 | 28,6% |
| Total afectado | 1.507,4 | 29,7% |

: Extension de la inundacion detectada con Sentinel-1 SAR (septiembre--octubre 2020). {#tbl-sar}

| Estacion | SAR diff (dB) | NDVI sept-2020 | Eventos historicos | Interpretacion |
|----------|--------------|----------------|-------------------|----------------|
| Isla Boqueron | -0,26 | -0,018 | 10 de 16 | Degradacion severa |
| Punta Cerro | -0,63 | -0,078 | 10 de 16 | Degradacion severa |
| Rio Sevilla | -0,76 | 0,050 | 10 de 16 | Estres hidrico |
| CP Pajarales | -1,71 | 0,300 | 9 de 16 | Inundacion bajo dosel |
| VIPIS | 0,12 | 0,350 | 9 de 16 | Sin afectacion mayor |
| Cano Palos | 0,15 | 0,400 | 2 de 16 | Sin afectacion mayor |
| Cano Clarin | -0,36 | 0,500 | 1 de 16 | Sin afectacion mayor |

: Cruce de senal SAR, NDVI y frecuencia de inundacion historica por estacion. {#tbl-cruce}

## Dinamica hidrica: transiciones JRC en zonas de manglar

El analisis de las transiciones del JRC Global Surface Water dentro de las zonas clasificadas como manglar revelo cambios hidricos de magnitud considerable. Se identificaron 59,8 km2 de agua estacional perdida ---areas que antes se inundaban estacionalmente y dejaron de hacerlo--- y 40,3 km2 de agua permanente perdida ---cuerpos de agua que se secaron dentro del manglar---, para un total de 100,1 km2 de perdida hidrica en zonas de manglar. Simultaneamente, 26,7 km2 presentaron nueva inundacion estacional y 155,7 km2 se clasificaron como efimero estacional, lo que indica que la dinamica hidrica del sistema se ha redistribuido espacialmente. Estos hallazgos permiten diferenciar dos mecanismos de degradacion que operan simultaneamente en la CGSM: la mortalidad por estres salino derivada de la perdida hidrica en las estaciones costeras orientales ---donde se concentran los 100,1 km2 de agua perdida--- y la inundacion prolongada bajo dosel en el Complejo de Pajarales ---donde se concentran los 155,7 km2 de inundacion efimera---.

| Transicion JRC en zona manglar | Area (km2) | Interpretacion |
|-------------------------------|-----------|----------------|
| Permanente perdido | 40,3 | Secamiento de cuerpos de agua |
| Estacional perdido | 59,8 | Zonas que dejaron de inundarse |
| Nuevo estacional | 26,7 | Nuevas areas de inundacion |
| Efimero estacional | 155,7 | Inundacion intermitente |
| Permanente a estacional | 9,8 | Transicion hidrologica |
| Efimero permanente | 13,8 | Agua intermitente persistente |

: Transiciones JRC Global Surface Water en zonas clasificadas como manglar. {#tbl-jrc}

## Ganancia y perdida de manglar (2020 vs 2024--2025)

| Categoria | Area (km2) |
|-----------|-----------|
| Perdida | 183,2 |
| Estable | 690,9 |
| Ganancia | 259,2 |
| Cambio neto | +76,0 |

: Ganancia y perdida de manglar entre el periodo de degradacion y el actual. {#tbl-cambio}

## Validacion contra Global Mangrove Watch v4.0

La clasificacion por umbrales espectrales se valido contra el Global Mangrove Watch v4.0 (GMW, ano 2020), que reporta 376,5 km2 de manglar en la CGSM. Se evaluaron multiples combinaciones de umbrales CMRI y NDVI, incorporando restricciones geofisicas de elevacion (SRTM < 10 m) y proximidad al agua (JRC, distancia < 3 km) para reducir la comision de falsos positivos en zonas de vegetacion terrestre fuera del habitat potencial del manglar. La optimizacion se realizo mediante muestreo aleatorio de 2.000 puntos sobre el AOI y calculo de la matriz de confusion contra el raster GMW. La mejor configuracion ---NDVI > 0,70, elevacion < 10 m, distancia al agua < 3 km--- alcanzo un F1-score de 0,442 con una precision global del 89,9%, clasificando 556 km2 como manglar potencial frente a los 376,5 km2 del GMW (ratio 1,48). La discrepancia residual se atribuye a tres factores: primero, el GMW emplea datos de resolucion gruesa (ALOS PALSAR, 25 m) que puede subestimar manglar disperso en zonas de transicion; segundo, los umbrales espectrales capturan vegetacion riberana y de humedal que comparte firma espectral con el manglar; y tercero, el GMW v4.0 corresponde al ano 2020, que coincide con el periodo de degradacion identificado en la Fase 2, por lo que la extension real del manglar puede haber sido menor ese ano.

| Metrica | Valor |
|---------|-------|
| Precision global (OA) | 0,899 |
| Precision (Producers) | 0,428 |
| Recall (Users) | 0,457 |
| F1-score | 0,442 |
| Area GMW 2020 | 376,5 km2 |
| Area clasificacion optimizada | 556,0 km2 |
| Ratio area (clasificacion / GMW) | 1,48 |
| Umbrales | NDVI > 0,70, elev < 10 m, agua < 3 km |

: Metricas de validacion de la clasificacion de manglar contra GMW v4.0 (2020). {#tbl-validacion}

## Dashboard interactivo

Se genero un dashboard interactivo en formato HTML (2,7 MB) mediante geemap, que integra 15 capas tematicas: composites RGB por periodo, NDVI por periodo, mapa de cambio NDVI, clasificacion de manglar por periodo, ganancia/perdida de manglar, inundacion SAR (agua abierta y bajo dosel), poligono del area de estudio y estaciones de monitoreo. El dashboard permite activar y desactivar capas, ajustar transparencia y navegar el mapa con zoom y desplazamiento, constituyendo una herramienta de consulta para gestores ambientales que no requiere software especializado ---solo un navegador web---.

# Conclusiones

El pipeline multilenguaje desarrollado permitio caracterizar la dinamica espaciotemporal de la cobertura de manglar en la CGSM durante el periodo 2013--2025, integrando analisis de series temporales de NDVI en Python (929 registros mensuales combinando Landsat 8 y Sentinel-2), deteccion de quiebres con bfast en R, segmentacion automatica con SamGeo en Python, metricas de fragmentacion en Julia, y validacion cruzada con datos SAR de inundacion y cartografia de referencia del Global Mangrove Watch. Los resultados principales son los siguientes: primero, la identificacion de un quiebre estructural generalizado en 2016 ---detectado por bfast en 7 de 8 estaciones---, asociado al evento El Nino 2015--2016, y de septiembre de 2020 como el segundo evento de mayor perturbacion ---NDVI negativo en 2 de 8 estaciones, z < -3---, asociado a La Nina 2020--2021, en un contexto historico de 14 eventos de inundacion documentados por la Global Flood Database entre 2001 y 2017; segundo, la deteccion de 18 anomalias significativas (z < -2) en la serie combinada de 12 anos, incluyendo 4 anomalias en VIPIS durante 2016 que solo fueron visibles al extender la serie con Landsat 8; tercero, una duplicacion del area de manglar clasificada entre el periodo de degradacion (4.938,5 ha) y el estado actual (8.444,5 ha) con incremento simultaneo del vigor vegetativo (NDVI de 0,503 a 0,564); cuarto, una reduccion de la densidad de parches de 11,19 a 6,47 por 1.000 hectareas, indicativa de consolidacion del paisaje; quinto, la diferenciacion mediante SAR de dos mecanismos de degradacion ---mortalidad por estres salino en estaciones costeras (100,1 km2 de agua perdida segun JRC) e inundacion bajo dosel en el Complejo de Pajarales (155,7 km2 de inundacion efimera)---, con un total de 1.507,4 km2 afectados durante el evento de 2020 (29,7% del AOI); y sexto, una validacion contra GMW v4.0 que arrojo un F1-score de 0,442 (OA = 0,899) con umbrales optimizados de NDVI > 0,70 y restricciones geofisicas de elevacion y proximidad al agua.

Entre las limitaciones tecnicas se identifican: la necesidad de remuestrear los composites RGB de 10 a 30 metros por restricciones de memoria para SamGeo (vit_b), lo que reduce la resolucion de la segmentacion; la diferencia en la respuesta espectral entre Landsat 8 y Sentinel-2, que introduce una discontinuidad en la serie combinada alrededor de 2018 ---visible en estaciones como VIPIS (NDVI medio L8 = 0,090 vs S2 = 0,353)--- que podria influir en la deteccion de quiebres por bfast; el F1-score de 0,442 que indica una sobreestimacion del area de manglar respecto al GMW, atribuible a la inclusion de vegetacion riberana con firma espectral similar; y la aproximacion del calculo de areas en Julia mediante conversion de coordenadas geograficas a metros. Se recomienda para trabajo futuro acotar el area de estudio a los poligonos oficiales del SFF CGSM y la Via Parque Isla de Salamanca, aplicar armonizacion espectral entre Landsat y Sentinel-2 para eliminar la discontinuidad en la serie, e implementar un clasificador Random Forest entrenado con puntos GMW para mejorar la precision de la clasificacion. No obstante, el pipeline demuestra la viabilidad de un enfoque GeoAI multilenguaje para el monitoreo costero y constituye un prototipo reproducible que puede ser adaptado a otros sistemas lagunares tropicales. El dashboard interactivo generado atiende directamente el requerimiento de monitoreo espacial sistematico del Plan de Manejo Ambiental del sitio Ramsar CGSM (Comision Conjunta CGSM, 2026).

# Referencias

- Beltran, J., Rodriguez, J. C., Carbono, E., y Blanco, J. (2022). Datos de monitoreo de la estructura de los manglares de la Cienaga Grande de Santa Marta (Magdalena) [Dataset]. INVEMAR. https://doi.org/10.15472/0fqdp4
- Bunting, P., Rosenqvist, A., Hilarides, L., Lucas, R. M., Thomas, T., Tadono, T., Worthington, T. A., Spalding, M., Murray, N. J., y Rebelo, L.-M. (2022). Global Mangrove Extent Change 1996--2020: Global Mangrove Watch Version 3.0. Remote Sensing, 14(15), 3657. https://doi.org/10.3390/rs14153657
- Comision Conjunta CGSM. (2026). Plan de Manejo Ambiental del sitio Ramsar Sistema Delta Estuarino del Rio Magdalena, Cienaga Grande de Santa Marta. CORPAMAG, Parques Nacionales Naturales, INVEMAR, MinAmbiente.
- Deng, T., Zhang, K., y Shen, Z.-J. (2021). A systematic review of a digital twin city: A new pattern of urban governance toward smart cities. Journal of Management Science and Engineering, 6(2), 125--134.
- Giri, C., Ochieng, E., Tieszen, L. L., Zhu, Z., Singh, A., Loveland, T., Masek, J., y Duke, N. (2011). Status and distribution of mangrove forests of the world using earth observation satellite data. Global Ecology and Biogeography, 20(1), 154--159.
- Gupta, K., Mukhopadhyay, A., Giri, S., Chanda, A., Datta Majumdar, S., Samanta, S., Mitra, D., Samal, R. N., Pattnaik, A. K., y Hazra, S. (2018). An index for discrimination of mangroves from non-mangroves using LANDSAT 8 OLI imagery. MethodsX, 5, 1129--1139.
- Hamilton, S. E., y Casey, D. (2016). Creation of a high spatio-temporal resolution global database of continuous mangrove forest cover for the 21st century (CGMFC-21). Global Ecology and Biogeography, 25(6), 729--738.
- Instituto de Investigaciones Marinas y Costeras [INVEMAR]. (2020). Cartografia de manglares de Colombia a escala 1:25.000 para Caribe y Pacifico. Procesamiento digital de imagenes en Google Earth Engine.
- Instituto de Investigaciones Marinas y Costeras [INVEMAR]. (2024). Monitoreo de las condiciones ambientales y los cambios estructurales y funcionales de las comunidades vegetales y de los recursos pesqueros durante la rehabilitacion de la Cienaga Grande de Santa Marta. Informe Tecnico Final (Vol. 23). https://www.invemar.org.co/inf-cgsm
- Raza, S., Zhang, J., Zuo, S., y Chen, J. (2024). Time series monitoring and analysis of Pakistans mangrove using Sentinel-2 data. Frontiers in Environmental Science, 12, 1416450. https://doi.org/10.3389/fenvs.2024.1416450
- Selvaraj, J. J., y Gallego-Perez, J. (2023). An enhanced approach to mangrove forest analysis in the Colombian Pacific coast using optical and SAR data in Google Earth Engine. Estuarine, Coastal and Shelf Science, 283, 108253. https://doi.org/10.1016/j.ecss.2023.108253
- Wu, Q., y Osco, L. (2023). samgeo: A Python package for segmenting geospatial data with the Segment Anything Model (SAM). Journal of Open Source Software, 8(89), 5663. https://doi.org/10.21105/joss.05663

# Anexo A: Configuracion del entorno Docker

El proyecto se ejecuta sobre el contenedor Docker sig_unal v1.11 proporcionado para la asignatura, con las siguientes modificaciones:
```bash
# Librerias Python adicionales instaladas
pip install earthengine-api geemap segment-geospatial leafmap
pip install rasterio geopandas xarray shapely folium rasterstats
pip install matplotlib pandas numpy scikit-learn contextily

# Autenticacion Google Earth Engine
earthengine authenticate --auth_mode=notebook
```
```r
# Paquetes R adicionales
install.packages(c("bfast", "terra", "sf", "ggplot2", "tseries"))
```
```julia
# Paquetes Julia adicionales
using Pkg
Pkg.add(["GeoJSON", "DataFrames", "CSV", "Statistics"])
```

La autenticacion con GEE se realiza mediante el flujo notebook, que genera un token almacenado en ~/.config/earthengine/credentials. El contenedor requiere un minimo de 8 GB de RAM asignados en Docker Desktop para ejecutar SamGeo con el backbone vit_b; con vit_h se requieren al menos 12 GB. Los composites RGB se remuestrean de 10 a 30 metros antes de la segmentacion para reducir el consumo de memoria.

# Anexo B: Estructura del repositorio GitHub

El repositorio esta disponible en https://github.com/LinaQuinteroF/proyecto-cgsm bajo licencia MIT.

- notebooks/01_gee_acquisition.ipynb: Fase 1 datacube Sentinel-2 + Landsat 8
- notebooks/02_time_series.ipynb: Fase 2 series temporales y z-scores (Python)
- notebooks/02b_bfast_ndvi.R.ipynb: Fase 2 deteccion de quiebres con bfast (R)
- notebooks/03_segmentation.ipynb: Fase 3 segmentacion SamGeo + validacion GMW
- notebooks/04_fragmentation.ipynb: Fase 4 metricas de fragmentacion (Julia)
- notebooks/05_flooding_nasa.ipynb: Validacion NASA SAR + Global Flood Database + JRC
- notebooks/06_dashboard.ipynb: Fase 5 dashboard interactivo
- outputs/tables/: 17 archivos CSV con resultados numericos
- outputs/figures/: 13 graficos y mapas estaticos (PNG)
- outputs/maps/: Dashboard HTML interactivo (2,7 MB)
- docs/informe_final.qmd: Informe Quarto reproducible
- docs/informe_final.html: Informe renderizado HTML
- docs/informe_final.pdf: Informe renderizado PDF

Los datos procesados (GeoTIFFs, GeoJSONs) no se incluyen en el repositorio por su tamano; se generan ejecutando los notebooks en orden secuencial con acceso a GEE autenticado.
"""

# Append part2 to existing file
with open('/home/rstudio/work/proyecto-cgsm/docs/informe_final.qmd', 'a') as f:
    f.write(part2)

import os
size = os.path.getsize('/home/rstudio/work/proyecto-cgsm/docs/informe_final.qmd')
print(f"QMD completo: {size} bytes ({size/1024:.0f} KB)")
with open('/home/rstudio/work/proyecto-cgsm/docs/informe_final.qmd') as f:
    lines = f.readlines()
print(f"Lineas: {len(lines)}")

QMD completo: 44485 bytes (43 KB)
Lineas: 396
